<a href="https://colab.research.google.com/github/camiloascencio-hash/GP-S/blob/main/Entrega2_BdM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Caso integrador Bebidas del Maipo S.A. — Entrega 2 · Cuaderno 0: puente E1 → E2
**Gestión de Producción de Bienes y Servicios · USACH · 2.º semestre 2026 · Grupo 7**

Este cuaderno **no formula ni resuelve el plan**. Solo deja los datos listos para empezar la Entrega 2:

1. Lee los **traspasos de la Entrega 1** desde `resultados/` del repositorio (salida de `Entrega1_BdM_v2.ipynb`) y
   verifica que estén completos y que sean los corregidos.
2. Lee los **datos de planificación** de `datos/`, revisa su consistencia y deja en una bitácora lo que no cuadra o
   falta.
3. Junta todo en **tablas ordenadas** (por SKU, por SKU y semana, por recurso y semana, y de preparaciones) en
   `resultados_E2/`, para que el modelo las lea directamente.

No se toma ninguna decisión de modelado: los valores son los de los datos, los del enunciado o los que la Entrega 1
dejó comprometidos en su tabla de traspaso.

| Sección | Contenido |
|---|---|
| 1 | Configuración y carga del repositorio |
| 2 | Traspasos de la E1 y su verificación |
| 3 | Datos de planificación y bitácora |
| 4 | Tablas para el modelo |
| 5 | Comprobaciones y exportación |

## 1. Configuración y carga del repositorio

In [ ]:
import os, json, hashlib, subprocess, platform, warnings
from importlib import metadata
import numpy as np
import pandas as pd

warnings.simplefilter("default")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

CONFIG = {
    "repo": "https://github.com/camiloascencio-hash/GP-S.git",
    "carpeta_datos": "GP-S/datos",
    "carpeta_E1": "GP-S/resultados",        # traspasos de la Entrega 1 (v2), tal como están en el repositorio
    "carpeta_salida": "resultados_E2",
    "T": 13,                                # horizonte táctico (semanas), enunciado §7.1
    "semana_focal": 12,                     # semana de los pedidos en firme (14/12/2026), enunciado §7.1
    # Datos del enunciado que no vienen en ningún CSV (§2.2 a §2.4)
    "costo_hora_cip_clp": 74_000,
    "costo_cambio_formato_clp": 38_000,
    "horas_cambio_formato": {"L1": 0.4, "L2": 0.6},
    "tol": 1e-6,
}
T, FOCAL, TOL = CONFIG["T"], CONFIG["semana_focal"], CONFIG["tol"]
SAL, E1, DAT = CONFIG["carpeta_salida"], CONFIG["carpeta_E1"], CONFIG["carpeta_datos"]

if not os.path.isdir(DAT):
    subprocess.run(["git", "clone", "-q", CONFIG["repo"]], check=True)
assert os.path.isdir(DAT) and os.path.isdir(E1), "No se encontró el repositorio"
os.makedirs(SAL, exist_ok=True)

versiones = {"python": platform.python_version(), **{p: metadata.version(p) for p in ["numpy", "pandas"]}}
print("Versiones:", versiones)

chequeos = []
def chequear(descripcion, condicion, detalle=""):
    assert bool(condicion), "FALLÓ: " + descripcion + (f" — {detalle}" if detalle else "")
    chequeos.append({"chequeo": descripcion, "resultado": "OK", "detalle": detalle})

def sha256(ruta):
    with open(ruta, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()

Versiones: {'python': '3.13.18', 'numpy': '2.4.6', 'pandas': '2.2.3'}


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 2. Traspasos de la Entrega 1 y su verificación

La Entrega 1 dejó comprometidos cuatro parámetros (§6.3): merma $\rho_\ell$ por línea (C1), jarabe extra por caja por
sobrellenado (C2), pronóstico $\hat d_{it}$ (C3) y desviación del error $\hat\sigma_{i,h}$ (C4), más el sesgo
$\hat b_{i,h}$ (C4b). Se usan **los valores de la tabla de traspaso consolidada**, que es la que la E1 entregó como
entrada de la E2. Se verifica que estén completos y que el σ̂ sea el corregido: si el repositorio tuviera el de la v1
(RMSE suavizado), el cuaderno se detiene. Se guarda el hash SHA-256 de cada archivo para rastrear cualquier cifra de
la E2 hasta el archivo exacto de la E1.

In [ ]:
archivos_E1 = ["traspaso_C1_merma.csv", "traspaso_C2_sobrellenado.csv", "traspaso_C3_pronostico.csv",
               "traspaso_C4_sigma.csv", "traspaso_C4b_sesgo.csv", "traspaso_consolidado.csv",
               "referencia_v1_rmse_suavizado.csv", "comprobaciones.csv", "config_ejecucion.json"]
for a in archivos_E1:
    chequear(f"Existe {a} en la carpeta de la E1", os.path.exists(os.path.join(E1, a)))

c3 = pd.read_csv(os.path.join(E1, "traspaso_C3_pronostico.csv"), index_col="t")
c4 = pd.read_csv(os.path.join(E1, "traspaso_C4_sigma.csv"), index_col="h")
c4b = pd.read_csv(os.path.join(E1, "traspaso_C4b_sesgo.csv"), index_col="h")
c4_v1 = pd.read_csv(os.path.join(E1, "referencia_v1_rmse_suavizado.csv"), index_col="h")
consolidado = pd.read_csv(os.path.join(E1, "traspaso_consolidado.csv"))
comp_E1 = pd.read_csv(os.path.join(E1, "comprobaciones.csv"))
with open(os.path.join(E1, "config_ejecucion.json"), encoding="utf-8") as f:
    config_E1 = json.load(f)

maestro = pd.read_csv(os.path.join(DAT, "maestro_sku.csv")).set_index("sku")
skus = list(maestro.index)

for nombre, tab in [("C3 pronóstico", c3), ("C4 σ̂", c4), ("C4b sesgo", c4b)]:
    chequear(f"{nombre}: 13 semanas × 9 SKU, sin vacíos", tab.shape == (T, len(skus)) and set(tab.columns) == set(skus)
             and list(tab.index) == list(range(1, T + 1)) and not tab.isna().any().any())
chequear("C3 pronóstico no negativo", (c3 >= 0).all().all())
chequear("C4 σ̂ finito y positivo", np.isfinite(c4.values).all() and (c4.values > 0).all())
dif_v1 = (c4[skus] - c4_v1[skus]).abs().max().max()
chequear("C4 es el σ̂ corregido (v2), no el RMSE suavizado de la v1", dif_v1 > 1, f"diferencia máxima con la v1: {dif_v1:.0f} cajas")
for p, tab in [("C3 pronóstico", c3), ("C4 sigma del error", c4), ("C4b sesgo del error", c4b)]:
    d_ = consolidado[consolidado["parametro"] == p].pivot(index="t_o_h", columns="sku", values="valor")
    chequear(f"Consolidado de la E1 = tabla {p.split()[0]} (117 registros)", len(consolidado[consolidado["parametro"] == p]) == 117
             and np.allclose(d_.loc[tab.index, skus].values, tab[skus].values, atol=TOL))
chequear("Las comprobaciones de la E1 están todas en OK", (comp_E1["resultado"] == "OK").all(), f"{len(comp_E1)} comprobaciones")

# Merma y sobrellenado: el valor que la E1 dejó en la tabla de traspaso consolidada
rho = consolidado[consolidado["parametro"] == "C1 rho (merma efectiva)"].set_index("linea")["valor"]
L_extra = consolidado[consolidado["parametro"] == "C2 jarabe extra por caja"].set_index("formato_L")["valor"]
L_extra.index = np.round(L_extra.index.astype(float), 1)
chequear("Merma ρ para L1 y L2, en (0, 1)", set(rho.index) == {"L1", "L2"} and rho.between(0, 1, inclusive="neither").all())
chequear("Jarabe extra por caja para los tres formatos", set(L_extra.index) == {0.5, 1.5, 3.0} and (L_extra >= 0).all())

hashes_E1 = {a: sha256(os.path.join(E1, a)) for a in archivos_E1}
print("Merma ρ por línea:", rho.round(4).to_dict())
print("Jarabe extra por caja (L), por formato:", L_extra.round(4).to_dict())
print("σ̂ medio por SKU (v2) frente a la v1, en cajas:")
display(pd.DataFrame({"σ̂ v2": c4[skus].mean(), "v1 (RMSE suavizado)": c4_v1[skus].mean()}).round(0).T)
print("Versiones de la ejecución de la E1:", config_E1.get("versiones"))

Merma ρ por línea: {'L1': 0.0192, 'L2': 0.0311}
Jarabe extra por caja (L), por formato: {0.5: 0.0686, 1.5: 0.0279, 3.0: 0.0323}
σ̂ medio por SKU (v2) frente a la v1, en cajas:


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,NEC-D05,NEC-D15,NEC-D30,CIT-L05,CIT-L15,CIT-L30,FUN-B05,FUN-B15,FUN-B30
σ̂ v2,423.0,725.0,163.0,441.0,437.0,102.0,475.0,557.0,360.0
v1 (RMSE suavizado),474.0,727.0,180.0,460.0,685.0,119.0,608.0,687.0,354.0


Versiones de la ejecución de la E1: {'python': '3.13.18', 'numpy': '2.4.6', 'pandas': '2.2.3', 'scipy': '1.16.3', 'statsmodels': '0.15.0', 'matplotlib': '3.10.0'}


## 3. Datos de planificación y bitácora

Se usan cinco archivos de `datos/`. Los de calidad y la demanda histórica ya quedaron resumidos en los traspasos de
la E1. Cada revisión queda en la bitácora con archivo, chequeo, resultado y observación. Lo que falta o es ambiguo
**se deja anotado como pendiente**: se resuelve al formular el modelo.

In [ ]:
cap = pd.read_csv(os.path.join(DAT, "capacidad_horizonte.csv")).set_index("t")
inv = pd.read_csv(os.path.join(DAT, "inventario_inicial.csv")).set_index("sku")
cip = pd.read_csv(os.path.join(DAT, "setups_cip.csv"))
firmes_raw = pd.read_csv(os.path.join(DAT, "compromisos_semana_focal.csv"))
for nombre, d_ in [("maestro_sku", maestro), ("capacidad_horizonte", cap), ("inventario_inicial", inv),
                   ("setups_cip", cip), ("compromisos_semana_focal", firmes_raw)]:
    print(f"{nombre:26s} {d_.shape[0]:3d} filas x {d_.shape[1]:2d} columnas")

bitacora = []
def registrar(archivo, chequeo, resultado, observacion):
    bitacora.append({"archivo": archivo, "chequeo": chequeo, "resultado": resultado, "observacion": observacion})

# ---------- capacidad_horizonte
chequear("capacidad_horizonte: 13 semanas = horizonte de la E1", list(cap.index) == list(range(1, T + 1)))
registrar("capacidad_horizonte.csv", "Horizonte", f"{len(cap)} semanas, {cap['fecha_lunes'].iloc[0]} a {cap['fecha_lunes'].iloc[-1]}",
          "Coincide con el pronóstico de la E1 (t = 1 a 13)")
nominal = {"mezcla": 55, "L2": 48}
for rec, col_reg, col_man in [("mezcla", "horas_mezcla_regular", "mantencion_programada_h_mezcla"),
                              ("L2", "horas_L2_regular", "mantencion_programada_h_L2")]:
    semanas_man = cap.index[cap[col_man] > 0].tolist()
    ya_descontada = np.allclose(cap.loc[semanas_man, col_reg] + cap.loc[semanas_man, col_man], nominal[rec])
    registrar("capacidad_horizonte.csv", f"Mantenciones de {rec}",
              f"semanas {semanas_man}: {[float(v) for v in cap.loc[semanas_man, col_man]]} h; regular + mantención = {nominal[rec]} h",
              "Las horas regulares ya vienen descontadas: no hay que restar la mantención otra vez")
    chequear(f"Mantenciones de {rec} ya descontadas de las horas regulares", ya_descontada)
t13 = cap.loc[T]
registrar("capacidad_horizonte.csv", "Semana 13 (21/12/2026)", f"regular mezcla {t13['horas_mezcla_regular']:.0f} h, L1 "
          f"{t13['horas_L1_regular']:.0f} h, L2 {t13['horas_L2_regular']:.0f} h, sin mantención registrada",
          "Capacidad menor que la normal (55, 40 y 48 h); coincide con el feriado del viernes 25/12")
registrar("capacidad_horizonte.csv", "Costo de hora extra de L1",
          "No hay columna para L1 (solo L2 y mezcla) y el enunciado tampoco lo da, aunque permite hasta 8 h extra",
          "PENDIENTE: dato faltante, se define al formular el modelo")
for col, v in [("tasa_mezcla_L_h", 2300), ("cip_mezcla_horas", 1.2), ("costo_hora_extra_L2_clp", 168_000), ("costo_hora_extra_mezcla_clp", 121_000),
               ("horas_mezcla_extra_max", 10), ("horas_L1_extra_max", 8), ("horas_L2_extra_max", 16)]:
    ok = (cap[col] == v).all()
    registrar("capacidad_horizonte.csv", f"{col} constante e igual al enunciado ({v})", str(ok), "OK" if ok else "Se usa el valor del CSV")
    chequear(f"{col} coincide con el enunciado", ok)

# ---------- maestro_sku
enunciado_maestro = {"NEC-D05": (3840, 6620), "NEC-D15": (5760, 9930), "NEC-D30": (7680, 13240), "CIT-L05": (2820, 5320),
                     "CIT-L15": (4230, 7980), "CIT-L30": (5640, 10640), "FUN-B05": (4860, 9920), "FUN-B15": (7290, 14880),
                     "FUN-B30": (9720, 19840)}
ok = all(tuple(maestro.loc[s, ["costo_variable_caja_clp", "precio_lista_caja_clp"]]) == v for s, v in enunciado_maestro.items())
registrar("maestro_sku.csv", "Costos y precios = tabla del enunciado §2.1", str(ok), "OK")
chequear("maestro_sku coincide con el enunciado", ok)
ok = (maestro["linea_envasado"] == np.where(maestro["formato_L"] == 0.5, "L1", "L2")).all()
registrar("maestro_sku.csv", "Línea según formato (0,5 L → L1; 1,5 y 3,0 L → L2)", str(ok), "OK")
chequear("Línea de envasado coherente con el formato", ok)
ok = (maestro["tasa_envasado_cajas_h"] == maestro["formato_L"].map({0.5: 180, 1.5: 210, 3.0: 130})).all()
registrar("maestro_sku.csv", "Tasas de envasado = enunciado §2.2 (180, 210 y 130 cajas/h)", str(ok), "OK")
chequear("Tasas de envasado coinciden con el enunciado", ok)
registrar("maestro_sku.csv", "Costo de inventario y penalidad de atraso",
          f"{sorted(int(v) for v in maestro['costo_inventario_caja_semana_clp'].unique())} y "
          f"{sorted(int(v) for v in maestro['penalidad_atraso_caja_semana_clp'].unique())} CLP/caja·semana",
          "Iguales para todos los SKU, como el enunciado §2.4")
registrar("enunciado §2.3", "Cambio de formato en L1 (0,4 h)", "L1 envasa un solo formato (0,5 L), y cada familia tiene un solo SKU en L1",
          "PENDIENTE: el dato existe, pero en L1 no hay dos formatos de una misma familia entre los cuales cambiar")

# ---------- setups_cip
H_cip = cip.pivot(index="familia_origen", columns="familia_destino", values="horas_limpieza_cip")
C_cip = cip.pivot(index="familia_origen", columns="familia_destino", values="costo_cip_clp")
familias = list(H_cip.index)
ok_costo = np.allclose(cip["costo_cip_clp"], cip["horas_limpieza_cip"] * CONFIG["costo_hora_cip_clp"])
asim = [(a, b) for a in familias for b in familias if a < b and H_cip.loc[a, b] != H_cip.loc[b, a]]
registrar("setups_cip.csv", "Costo = horas × 74.000 (enunciado §2.3)", str(ok_costo), "OK")
registrar("setups_cip.csv", "Diagonal nula y asimetría", f"diagonal = 0: {bool((np.diag(H_cip) == 0).all())}; pares asimétricos: {asim}",
          "Matriz completa disponible para el modelo")
chequear("Costo CIP = horas × 74.000 y diagonal nula", ok_costo and (np.diag(H_cip) == 0).all())
chequear("Matriz CIP = enunciado §2.3", np.allclose(H_cip.loc[["CIT", "FUN", "NEC"], ["CIT", "FUN", "NEC"]].values,
                                                    [[0, 1.0, 1.0], [2.4, 0, 1.4], [2.8, 2.8, 0]]))

# ---------- inventario_inicial
chequear("inventario_inicial: los 9 SKU", set(inv.index) == set(skus))
registrar("inventario_inicial.csv", "Lead time de jarabe", f"{sorted(int(v) for v in inv['lead_time_jarabe_semanas'].unique())} semana(s) en los 9 SKU",
          "PENDIENTE: el enunciado no dice si es entre mezcla y envasado o de abastecimiento; se define al formular")
registrar("inventario_inicial.csv", "Nivel de servicio objetivo",
          f"{ {s: float(v) for s, v in inv['nivel_servicio_objetivo'].items()} }", "95 % en CIT y NEC, 92 % en FUN")
registrar("inventario_inicial.csv", "Atraso inicial", "El archivo no trae atraso pendiente al inicio del horizonte", "Solo inventario inicial")

# ---------- compromisos_semana_focal
ok = (firmes_raw["linea_envasado"] == firmes_raw["sku"].map(maestro["linea_envasado"])).all() and \
     (firmes_raw["familia"] == firmes_raw["sku"].map(maestro["familia"])).all()
registrar("compromisos_semana_focal.csv", "Línea y familia de cada orden = maestro", str(ok), "OK")
chequear("Pedidos en firme coherentes con el maestro", ok)
registrar("compromisos_semana_focal.csv", "Detalle por día y hora",
          f"{len(firmes_raw)} órdenes, días {sorted(int(v) for v in firmes_raw['dia_comprometido'].unique())}",
          "La E2 usa la suma semanal por SKU (§7.1, elemento 8); día y hora son para la E3")

maestro_sku                  9 filas x 12 columnas
capacidad_horizonte         13 filas x 13 columnas
inventario_inicial           9 filas x  4 columnas
setups_cip                   9 filas x  5 columnas
compromisos_semana_focal    15 filas x 10 columnas


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 4. Tablas para el modelo

### 4.1 Por SKU

Datos del maestro y del inventario inicial, más los dos traspasos por línea y formato de la E1. Las únicas columnas
calculadas son conversiones de unidades: horas de envasado por caja ($1/\text{tasa}$), litros de jarabe por caja
(nominales + sobrellenado de C2) y horas de mezcla por caja (litros / 2.300).

In [ ]:
tasa_mezcla = float(cap["tasa_mezcla_L_h"].iloc[0])
par_sku = pd.DataFrame(index=pd.Index(skus, name="sku"))
par_sku["familia"] = maestro["familia"]
par_sku["linea"] = maestro["linea_envasado"]
par_sku["formato_L"] = maestro["formato_L"]
par_sku["botellas_por_caja"] = maestro["botellas_por_caja"]
par_sku["tasa_cajas_h"] = maestro["tasa_envasado_cajas_h"]
par_sku["h_envasado_por_caja"] = 1 / maestro["tasa_envasado_cajas_h"]
par_sku["litros_nominales_caja"] = maestro["litros_por_caja"]
par_sku["litros_sobrellenado_caja"] = maestro["formato_L"].round(1).map(L_extra)          # traspaso C2
par_sku["litros_jarabe_caja"] = par_sku["litros_nominales_caja"] + par_sku["litros_sobrellenado_caja"]
par_sku["h_mezcla_por_caja"] = par_sku["litros_jarabe_caja"] / tasa_mezcla
par_sku["rho_merma"] = par_sku["linea"].map(rho)                                          # traspaso C1
par_sku["inventario_inicial"] = inv["inventario_inicial_cajas"]
par_sku["lote_minimo_cajas"] = inv["lote_minimo_cajas"]
par_sku["nivel_servicio"] = inv["nivel_servicio_objetivo"]
par_sku["lead_time_jarabe_semanas"] = inv["lead_time_jarabe_semanas"]
par_sku["costo_variable_caja"] = maestro["costo_variable_caja_clp"]
par_sku["precio_lista_caja"] = maestro["precio_lista_caja_clp"]
par_sku["costo_inventario_caja_sem"] = maestro["costo_inventario_caja_semana_clp"]
par_sku["penalidad_atraso_caja_sem"] = maestro["penalidad_atraso_caja_semana_clp"]
display(par_sku.round(4))

,familia,linea,formato_L,botellas_por_caja,tasa_cajas_h,h_envasado_por_caja,litros_nominales_caja,litros_sobrellenado_caja,litros_jarabe_caja,h_mezcla_por_caja,rho_merma,inventario_inicial,lote_minimo_cajas,nivel_servicio,lead_time_jarabe_semanas,costo_variable_caja,precio_lista_caja,costo_inventario_caja_sem,penalidad_atraso_caja_sem
sku,,,,,,,,,,,,,,,,,,,
NEC-D05,NEC,L1,0.5,12,180,0.0056,6.0,0.0686,6.0686,0.0026,0.0192,1024,350,0.95,1,3840,6620,95,1400
NEC-D15,NEC,L2,1.5,6,210,0.0048,9.0,0.0279,9.0279,0.0039,0.0311,3162,600,0.95,1,5760,9930,95,1400
NEC-D30,NEC,L2,3.0,4,130,0.0077,12.0,0.0323,12.0323,0.0052,0.0311,339,140,0.95,1,7680,13240,95,1400
CIT-L05,CIT,L1,0.5,12,180,0.0056,6.0,0.0686,6.0686,0.0026,0.0192,1217,350,0.95,1,2820,5320,95,1400
CIT-L15,CIT,L2,1.5,6,210,0.0048,9.0,0.0279,9.0279,0.0039,0.0311,1282,430,0.95,1,4230,7980,95,1400
CIT-L30,CIT,L2,3.0,4,130,0.0077,12.0,0.0323,12.0323,0.0052,0.0311,255,70,0.95,1,5640,10640,95,1400
FUN-B05,FUN,L1,0.5,12,180,0.0056,6.0,0.0686,6.0686,0.0026,0.0192,2135,550,0.92,1,4860,9920,95,1400
FUN-B15,FUN,L2,1.5,6,210,0.0048,9.0,0.0279,9.0279,0.0039,0.0311,1261,360,0.92,1,7290,14880,95,1400
FUN-B30,FUN,L2,3.0,4,130,0.0077,12.0,0.0323,12.0323,0.0052,0.0311,155,30,0.92,1,9720,19840,95,1400


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


### 4.2 Por SKU y semana

Pronóstico $\hat d_{it}$ (C3), σ̂ y sesgo b̂ al horizonte $h = t$ (C4 y C4b) y pedidos en firme de la semana 12,
sumados por SKU. Todo en cajas vendibles.

In [ ]:
firmes = firmes_raw.groupby("sku")["cantidad_comprometida_cajas"].sum().reindex(skus, fill_value=0)
par_sku_sem = pd.DataFrame([{"sku": i, "t": t, "fecha_lunes": cap.loc[t, "fecha_lunes"], "demanda": c3.loc[t, i],
                             "sigma": c4.loc[t, i], "sesgo": c4b.loc[t, i],
                             "firmes": float(firmes[i]) if t == FOCAL else 0.0} for i in skus for t in range(1, T + 1)])
resumen_firmes = pd.DataFrame({"firmes semana 12": firmes, "pronóstico semana 12": c3.loc[FOCAL, skus],
                               "firmes / pronóstico %": 100 * firmes / c3.loc[FOCAL, skus]})
display(resumen_firmes.round(1))
chequear("Pedidos en firme ≤ pronóstico de la semana 12 en todos los SKU", (firmes <= c3.loc[FOCAL, skus] + TOL).all())
registrar("compromisos_semana_focal.csv", "Pedidos en firme frente al pronóstico de la semana 12",
          f"entre {resumen_firmes['firmes / pronóstico %'].min():.0f} % y {resumen_firmes['firmes / pronóstico %'].max():.0f} % del pronóstico",
          "Ningún SKU tiene más pedidos firmes que pronóstico")

,firmes semana 12,pronóstico semana 12,firmes / pronóstico %
NEC-D05,898,1341.7,66.9
NEC-D15,1320,2370.6,55.7
NEC-D30,313,548.0,57.1
CIT-L05,839,2585.3,32.5
CIT-L15,1026,2904.5,35.3
CIT-L30,157,568.8,27.6
FUN-B05,1340,2432.7,55.1
FUN-B15,860,1729.8,49.7
FUN-B30,66,226.0,29.2


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


### 4.3 Por recurso y semana

Horas regulares (ya sin mantención), máximo de horas extra y costo de la hora extra para la mezcla (MEZ) y las dos
líneas. El costo de la hora extra de L1 queda **vacío** porque no está en los datos (ver bitácora).

In [ ]:
filas = []
for t in cap.index:
    r = cap.loc[t]
    filas += [
        {"t": t, "fecha_lunes": r["fecha_lunes"], "recurso": "MEZ", "h_regular": r["horas_mezcla_regular"], "h_extra_max": r["horas_mezcla_extra_max"],
         "costo_hora_extra": r["costo_hora_extra_mezcla_clp"], "mantencion_h": r["mantencion_programada_h_mezcla"]},
        {"t": t, "fecha_lunes": r["fecha_lunes"], "recurso": "L1", "h_regular": r["horas_L1_regular"], "h_extra_max": r["horas_L1_extra_max"],
         "costo_hora_extra": np.nan, "mantencion_h": 0.0},
        {"t": t, "fecha_lunes": r["fecha_lunes"], "recurso": "L2", "h_regular": r["horas_L2_regular"], "h_extra_max": r["horas_L2_extra_max"],
         "costo_hora_extra": r["costo_hora_extra_L2_clp"], "mantencion_h": r["mantencion_programada_h_L2"]}]
par_cap = pd.DataFrame(filas)
display(par_cap.pivot(index="t", columns="recurso", values=["h_regular", "h_extra_max"]))

h_regular             h_extra_max            
recurso        L1    L2   MEZ          L1    L2   MEZ
t                                                    
1            40.0  48.0  55.0         8.0  16.0  10.0
2            40.0  48.0  55.0         8.0  16.0  10.0
3            40.0  48.0  55.0         8.0  16.0  10.0
4            40.0  40.0  55.0         8.0  16.0  10.0
5            40.0  48.0  55.0         8.0  16.0  10.0
6            40.0  48.0  49.0         8.0  16.0  10.0
7            40.0  48.0  55.0         8.0  16.0  10.0
8            40.0  48.0  55.0         8.0  16.0  10.0
9            40.0  40.0  55.0         8.0  16.0  10.0
10           40.0  48.0  55.0         8.0  16.0  10.0
11           40.0  48.0  55.0         8.0  16.0  10.0
12           40.0  48.0  55.0         8.0  16.0  10.0
13           36.0  40.0  47.0         8.0  16.0  10.0

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


### 4.4 Preparaciones

Matriz completa de CIP de envasado entre familias (horas y costo), CIP de mezcla entre familias (1,2 h) y cambio de
formato dentro de una familia (0,4 h en L1, 0,6 h en L2, 38.000 CLP). Se entregan tal como vienen; cómo representarlas
en el modelo se decide al formular.

In [ ]:
print("CIP de envasado (horas), fila = familia que sale, columna = familia que entra:")
display(H_cip)
par_prep = pd.DataFrame([
    {"concepto": "CIP de mezcla entre familias", "recurso": "MEZ", "horas": float(cap["cip_mezcla_horas"].iloc[0]),
     "costo_clp": float(cap["cip_mezcla_horas"].iloc[0]) * CONFIG["costo_hora_cip_clp"], "fuente": "capacidad_horizonte.csv y §2.3"},
    {"concepto": "Cambio de formato en la misma familia", "recurso": "L1", "horas": CONFIG["horas_cambio_formato"]["L1"],
     "costo_clp": CONFIG["costo_cambio_formato_clp"], "fuente": "enunciado §2.3"},
    {"concepto": "Cambio de formato en la misma familia", "recurso": "L2", "horas": CONFIG["horas_cambio_formato"]["L2"],
     "costo_clp": CONFIG["costo_cambio_formato_clp"], "fuente": "enunciado §2.3"},
    {"concepto": "Costo de una hora de CIP", "recurso": "todos", "horas": 1.0, "costo_clp": CONFIG["costo_hora_cip_clp"], "fuente": "enunciado §2.3 y §2.4"}])
display(par_prep)

CIP de envasado (horas), fila = familia que sale, columna = familia que entra:


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


familia_destino,CIT,FUN,NEC
familia_origen,,,
CIT,0.0,1.0,1.0
FUN,2.4,0.0,1.4
NEC,2.8,2.8,0.0


,concepto,recurso,horas,costo_clp,fuente
0,CIP de mezcla entre familias,MEZ,1.2,88800.0,capacidad_horizonte.csv y §2.3
1,Cambio de formato en la misma familia,L1,0.4,38000.0,enunciado §2.3
2,Cambio de formato en la misma familia,L2,0.6,38000.0,enunciado §2.3
3,Costo de una hora de CIP,todos,1.0,74000.0,enunciado §2.3 y §2.4


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 5. Comprobaciones y exportación

In [ ]:
chequear("Tabla por SKU: 9 SKU sin vacíos", len(par_sku) == len(skus) and not par_sku.isna().any().any())
chequear("Tabla SKU × semana: 117 registros sin duplicados ni vacíos",
         len(par_sku_sem) == 117 and not par_sku_sem.duplicated(["sku", "t"]).any() and not par_sku_sem.isna().any().any())
chequear("Demanda = C3 de la E1 (literal)", np.allclose(par_sku_sem.pivot(index="t", columns="sku", values="demanda")[skus].values, c3[skus].values, atol=TOL))
chequear("σ̂ = C4 de la E1 (literal)", np.allclose(par_sku_sem.pivot(index="t", columns="sku", values="sigma")[skus].values, c4[skus].values, atol=TOL))
chequear("Pedidos en firme solo en la semana 12 y suma = archivo",
         par_sku_sem.query("t != @FOCAL")["firmes"].eq(0).all() and abs(par_sku_sem["firmes"].sum() - firmes_raw["cantidad_comprometida_cajas"].sum()) < TOL)
chequear("Capacidad: 13 semanas × 3 recursos", len(par_cap) == 3 * T)

par_sku.to_csv(os.path.join(SAL, "datos_sku.csv"))
par_sku_sem.to_csv(os.path.join(SAL, "datos_sku_semana.csv"), index=False)
par_cap.to_csv(os.path.join(SAL, "datos_capacidad.csv"), index=False)
H_cip.to_csv(os.path.join(SAL, "matriz_cip_envasado_horas.csv"))
C_cip.to_csv(os.path.join(SAL, "matriz_cip_envasado_costo.csv"))
par_prep.to_csv(os.path.join(SAL, "datos_preparaciones.csv"), index=False)
firmes_raw.to_csv(os.path.join(SAL, "pedidos_firmes_semana12_detalle.csv"), index=False)
pd.DataFrame(bitacora).to_csv(os.path.join(SAL, "bitacora_datos_E2.csv"), index=False)

hashes_datos = {a: sha256(os.path.join(DAT, a)) for a in ["maestro_sku.csv", "capacidad_horizonte.csv", "inventario_inicial.csv",
                                                          "setups_cip.csv", "compromisos_semana_focal.csv"]}
with open(os.path.join(SAL, "config_puente.json"), "w", encoding="utf-8") as f:
    json.dump({"config": CONFIG, "versiones": versiones, "hash_traspasos_E1": hashes_E1, "hash_datos": hashes_datos,
               "versiones_ejecucion_E1": config_E1.get("versiones"), "tasa_mezcla_L_h": tasa_mezcla,
               "familias": familias, "skus": skus}, f, indent=2, ensure_ascii=False, default=str)

tabla_chequeos = pd.DataFrame(chequeos)
tabla_chequeos.to_csv(os.path.join(SAL, "comprobaciones_puente.csv"), index=False)
print("Bitácora de datos:")
display(pd.DataFrame(bitacora))
print("Pendientes para la formulación:")
display(pd.DataFrame(bitacora).query("observacion.str.startswith('PENDIENTE')")[["archivo", "chequeo", "observacion"]])
display(tabla_chequeos)
print(f"{len(chequeos)} comprobaciones superadas.")
print("Archivos en", SAL + "/:", sorted(os.listdir(SAL)))

Bitácora de datos:


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,archivo,chequeo,resultado,observacion
0,capacidad_horizonte.csv,Horizonte,"13 semanas, 2026-09-28 a 2026-12-21",Coincide con el pronóstico de la E1 (t = 1 a 13)
1,capacidad_horizonte.csv,Mantenciones de mezcla,semanas [6]: [6.0] h; regular + mantención = 55 h,Las horas regulares ya vienen descontadas: no ...
2,capacidad_horizonte.csv,Mantenciones de L2,"semanas [4, 9]: [8.0, 8.0] h; regular + manten...",Las horas regulares ya vienen descontadas: no ...
3,capacidad_horizonte.csv,Semana 13 (21/12/2026),"regular mezcla 47 h, L1 36 h, L2 40 h, sin man...","Capacidad menor que la normal (55, 40 y 48 h);..."
4,capacidad_horizonte.csv,Costo de hora extra de L1,No hay columna para L1 (solo L2 y mezcla) y el...,"PENDIENTE: dato faltante, se define al formula..."
5,capacidad_horizonte.csv,tasa_mezcla_L_h constante e igual al enunciado...,True,OK
6,capacidad_horizonte.csv,cip_mezcla_horas constante e igual al enunciad...,True,OK
7,capacidad_horizonte.csv,costo_hora_extra_L2_clp constante e igual al e...,True,OK
8,capacidad_horizonte.csv,costo_hora_extra_mezcla_clp constante e igual ...,True,OK
9,capacidad_horizonte.csv,horas_mezcla_extra_max constante e igual al en...,True,OK


Pendientes para la formulación:


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,archivo,chequeo,observacion
4,capacidad_horizonte.csv,Costo de hora extra de L1,"PENDIENTE: dato faltante, se define al formula..."
16,enunciado §2.3,"Cambio de formato en L1 (0,4 h)","PENDIENTE: el dato existe, pero en L1 no hay d..."
19,inventario_inicial.csv,Lead time de jarabe,PENDIENTE: el enunciado no dice si es entre me...


,chequeo,resultado,detalle
0,Existe traspaso_C1_merma.csv en la carpeta de ...,OK,
1,Existe traspaso_C2_sobrellenado.csv en la carp...,OK,
2,Existe traspaso_C3_pronostico.csv en la carpet...,OK,
3,Existe traspaso_C4_sigma.csv en la carpeta de ...,OK,
4,Existe traspaso_C4b_sesgo.csv en la carpeta de...,OK,
5,Existe traspaso_consolidado.csv en la carpeta ...,OK,
6,Existe referencia_v1_rmse_suavizado.csv en la ...,OK,
7,Existe comprobaciones.csv en la carpeta de la E1,OK,
8,Existe config_ejecucion.json en la carpeta de ...,OK,
9,"C3 pronóstico: 13 semanas × 9 SKU, sin vacíos",OK,


45 comprobaciones superadas.
Archivos en resultados_E2/: ['bitacora_datos_E2.csv', 'comprobaciones_puente.csv', 'config_puente.json', 'datos_capacidad.csv', 'datos_preparaciones.csv', 'datos_sku.csv', 'datos_sku_semana.csv', 'matriz_cip_envasado_costo.csv', 'matriz_cip_envasado_horas.csv', 'pedidos_firmes_semana12_detalle.csv']


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Qué queda listo

| Archivo | Contenido |
|---|---|
| `datos_sku.csv` | Por SKU: familia, línea, formato, tasa y horas de envasado por caja, litros y horas de mezcla por caja (con sobrellenado), merma ρ, inventario inicial, lote mínimo, nivel de servicio, lead time y costos |
| `datos_sku_semana.csv` | Por SKU y semana: pronóstico, σ̂, sesgo y pedidos en firme (semana 12) |
| `datos_capacidad.csv` | Por recurso y semana: horas regulares, máximo de extra, costo de la hora extra y mantención |
| `matriz_cip_envasado_horas.csv`, `matriz_cip_envasado_costo.csv`, `datos_preparaciones.csv` | CIP de envasado entre familias, CIP de mezcla y cambio de formato |
| `pedidos_firmes_semana12_detalle.csv` | Las 15 órdenes en firme con día y hora (para la E3) |
| `bitacora_datos_E2.csv` | Revisión de los datos, con los pendientes marcados |
| `config_puente.json`, `comprobaciones_puente.csv` | Hashes de los archivos de entrada, versiones y comprobaciones |

## 6. Modelo de lot-sizing capacitado — Parte A: formulación

El modelo se construye elemento por elemento, en el orden de §7.1. Cada elemento lee las tablas de
`resultados_E2/` y agrega sus variables y restricciones al mismo problema `modelo`.

### 6.1 Elemento 1: balance de inventario con atraso penalizado y rendimiento $\rho_\ell$

**Punto de partida (Unidad 4, «Backlogging o demanda atrasada»).** Con inventario $s^i_t$ y atraso $b^i_t$ al cierre de
cada semana, el balance de la clase es

$$s^i_{t-1} + x^i_t + b^i_t = d^i_t + s^i_t + b^i_{t-1} \qquad \forall i, t,$$

y el atraso entra en la función objetivo con su penalización $\pi^i_t\,b^i_t$.

**Lo que agrega BdM: el rendimiento.** La Entrega 1 dejó la merma efectiva $\rho_\ell$ por línea: de cada caja que se
envasa en la línea $\ell$, solo $1-\rho_\ell$ es vendible (§6.3: «producir $x/(1-\rho_\ell)$ para obtener $x$ cajas
vendibles»). Se define $x^i_t$ como las **cajas envasadas** (brutas), porque son las que consumen horas de línea y litros
de jarabe, incluidas las defectuosas. Al inventario solo entra la parte vendible:

$$s^i_{t-1} + (1-\rho_{\ell(i)})\,x^i_t + b^i_t = d^i_t + s^i_t + b^i_{t-1} \qquad \forall i \in I,\; t = 1,\dots,13$$

**Conjuntos y datos.**

| Símbolo | Significado | Fuente |
|---|---|---|
| $I$ | 9 SKU; $\ell(i)$ es su línea de envasado (L1 para 0,5 L, L2 para 1,5 y 3,0 L) | `datos_sku.csv` |
| $t = 1,\dots,13$ | Semanas del horizonte (t = 1 es la semana del 28/09/2026) | `datos_sku_semana.csv` |
| $d^i_t$ | Pronóstico de demanda, cajas vendibles (traspaso C3) | `datos_sku_semana.csv` |
| $\rho_\ell$ | Merma efectiva: 1,92 % en L1 y 3,11 % en L2 (traspaso C1) | `datos_sku.csv` |
| $s^i_0$ | Inventario al cierre de la semana 156 | `datos_sku.csv` |
| $b^i_0$ | Atraso al inicio: 0, porque `inventario_inicial.csv` no registra atraso pendiente | bitácora |
| $h_i$ | Costo de mantener inventario: 95 CLP por caja y semana | `datos_sku.csv` |
| $\pi_i$ | Penalidad por demanda atrasada: 1.400 CLP por caja y semana | `datos_sku.csv` |

**Variables.** $x^i_t \ge 0$ cajas envasadas; $s^i_t \ge 0$ inventario vendible al cierre; $b^i_t \ge 0$ demanda atrasada
al cierre. $s$, $b$ y $d$ están en cajas vendibles; $x$ en cajas envasadas.

**Términos de la función objetivo que aporta este elemento.**

$$\sum_{i \in I}\sum_{t=1}^{13} \left(h_i\, s^i_t + \pi_i\, b^i_t\right)$$

**Cómo se lee.** Lo que había al inicio de la semana, más lo vendible que se envasa, más lo que se deja pendiente,
tiene que alcanzar para la demanda de la semana, el inventario que pasa a la siguiente y lo atrasado que venía de
antes. Como la penalidad (1.400) es casi 15 veces el costo de inventario (95), atrasar una caja una semana cuesta lo
mismo que guardarla casi 15 semanas: el modelo solo atrasará cuando no tenga capacidad para adelantar.

**Lo que este elemento deja abierto (se resuelve en los elementos siguientes).**
- **Atraso al final del horizonte $b^i_{13}$.** El balance permite terminar con demanda pendiente; en ese caso la
  penalidad se paga una sola semana y la demanda «desaparece». Se trata con la restricción de fin de horizonte
  (elemento 7).
- **Stock de seguridad** (elemento 4) y **lead time de jarabe** (elemento 6). El balance usa $x^i_t$ en la misma semana:
  si el lead time termina afectando al envasado y no solo a la mezcla, se cambiará $x^i_t$ por $x^i_{t-\gamma}$, como en la
  extensión de lead time de la clase.

In [2]:
# ============================================================
# ELEMENTO 1 — BALANCE DE INVENTARIO CON ATRASO Y RENDIMIENTO
# ============================================================
import os
import numpy as np
import pandas as pd

# ---- 1. Instalar PuLP --------------------------------------
# PuLP no viene instalado en Colab. Se fija la versión 2.9.0 porque trae el solver CBC
# incluido y usa la sintaxis clásica (LpVariable.dicts, LpStatus). PuLP 4 cambió esa
# sintaxis y ya no incluye CBC, así que este código no correría con esa versión.
import importlib.util, importlib.metadata, subprocess, sys
VERSION_PULP = "2.9.0"
if importlib.util.find_spec("pulp") is None or importlib.metadata.version("pulp") != VERSION_PULP:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"pulp=={VERSION_PULP}"], check=True)
import pulp
assert pulp.__version__ == VERSION_PULP, f"Se esperaba PuLP {VERSION_PULP} y hay {pulp.__version__}: reinicia el entorno y vuelve a ejecutar"
SOLVER = pulp.PULP_CBC_CMD(msg=False)          # CBC sin imprimir el log del solver
print("PuLP", pulp.__version__, "| solver:", SOLVER.name)

# ---- 2. Leer los datos del puente --------------------------
# Usa la carpeta resultados_E2 de esta sesión (si se ejecutó el puente arriba)
# o, si no existe, la que está subida en el repositorio.
CARPETA_E2 = "resultados_E2" if os.path.isdir("resultados_E2") else "GP-S/resultados_E2"
datos_sku = pd.read_csv(os.path.join(CARPETA_E2, "datos_sku.csv"), index_col="sku")
datos_sku_sem = pd.read_csv(os.path.join(CARPETA_E2, "datos_sku_semana.csv"))

# ---- 3. Conjuntos y parámetros -----------------------------
I = list(datos_sku.index)                                   # conjunto I: los 9 SKU
TT = sorted(datos_sku_sem["t"].unique())                    # semanas t = 1..13
d = datos_sku_sem.pivot(index="sku", columns="t", values="demanda").loc[I, TT]   # d[i, t]: pronóstico (cajas vendibles, C3)
rend = 1 - datos_sku["rho_merma"]                           # 1 − ρ_ℓ(i): fracción vendible de cada caja envasada (C1)
s0 = datos_sku["inventario_inicial"]                        # s[i, 0]: inventario al cierre de la semana 156
b0 = pd.Series(0.0, index=I)                                # b[i, 0]: atraso inicial = 0 (no hay atraso registrado)
h = datos_sku["costo_inventario_caja_sem"]                  # h_i = 95 CLP por caja y semana
pi = datos_sku["penalidad_atraso_caja_sem"]                 # π_i = 1.400 CLP por caja y semana
assert len(I) == 9 and TT == list(range(1, 14)), "Revisar las tablas de resultados_E2"

# ---- 4. Variables y restricción de balance -----------------
def crear_variables(problema, nombre, indices, **opciones):
    """Crea un diccionario de variables indexadas por (SKU, semana): var[i][t]."""
    return pulp.LpVariable.dicts(nombre, indices, **opciones)

def agregar_balance(problema):
    """Agrega al problema las variables x, s, b y el balance de inventario de cada SKU y semana.
    Se escribe como función para poder usarla en el modelo y en la prueba sin repetir código."""
    x = crear_variables(problema, "x", (I, TT), lowBound=0)     # x[i][t]: cajas ENVASADAS (brutas)
    s = crear_variables(problema, "s", (I, TT), lowBound=0)     # s[i][t]: inventario vendible al cierre de t
    b = crear_variables(problema, "b", (I, TT), lowBound=0)     # b[i][t]: demanda atrasada al cierre de t
    for i in I:
        for t in TT:
            # En t = 1 el inventario y el atraso "anteriores" son datos (s0, b0); después, son variables
            s_ant = float(s0[i]) if t == TT[0] else s[i][t - 1]
            b_ant = float(b0[i]) if t == TT[0] else b[i][t - 1]
            # s[t-1] + (1 − ρ)·x[t] + b[t] = d[t] + s[t] + b[t-1]
            problema += (s_ant + float(rend[i]) * x[i][t] + b[i][t] == float(d.loc[i, t]) + s[i][t] + b_ant), f"balance_{i}_{t}"
    # Términos de costo que aporta este elemento a la función objetivo
    costo_inventario = pulp.lpSum(float(h[i]) * s[i][t] for i in I for t in TT)    # Σ h·s
    costo_atraso = pulp.lpSum(float(pi[i]) * b[i][t] for i in I for t in TT)       # Σ π·b
    return x, s, b, {"inventario": costo_inventario, "atraso": costo_atraso}

# ---- 5. Crear el modelo y agregarle el elemento 1 ----------
# Los elementos siguientes (2 a 8) agregan sus restricciones a este mismo "modelo"
# y sus costos al diccionario "terminos_objetivo".
modelo = pulp.LpProblem("BdM_lot_sizing_capacitado", pulp.LpMinimize)
x, s, b, terminos_objetivo = agregar_balance(modelo)

print(f"Elemento 1: {len(I) * len(TT)} restricciones de balance, {3 * len(I) * len(TT)} variables (x, s, b)")
display(pd.DataFrame({"línea": datos_sku["linea"], "ρ": datos_sku["rho_merma"], "rendimiento 1 − ρ": rend,
                      "cajas a envasar por 100 vendibles": 100 / rend, "inventario inicial s₀": s0}).round(4))

PuLP 2.9.0 | solver: PULP_CBC_CMD


FileNotFoundError: [Errno 2] No such file or directory: 'GP-S/resultados_E2/datos_sku.csv'

**Prueba del elemento 1.** Antes de agregar capacidad, se resuelve solo con el balance para comprobar que está bien
escrito. Sin ninguna otra restricción el óptimo se conoce de antemano: no atrasar, gastar primero el inventario inicial
y después envasar cada semana justo la demanda dividida por el rendimiento. El único costo es mantener el inventario
inicial mientras dura: $\sum_{i,t} h_i \max\left(0,\; s^i_0 - \sum_{\tau \le t} d^i_\tau\right)$. Para que el atraso final no
«desaparezca» demanda en esta prueba, se fija $b^i_{13} = 0$ solo aquí (el modelo lo resuelve en el elemento 7). Se
verifica que el solver llegue a ese costo, que la producción vendible sea la demanda neta y que el balance cierre en
las 117 celdas (con la tolerancia numérica del solver).

In [ ]:
# ============================================================
# PRUEBA DEL ELEMENTO 1
# ============================================================
# Se arma un problema APARTE ("prueba") con solo el balance, para no tocar el modelo.
prueba = pulp.LpProblem("prueba_elemento_1", pulp.LpMinimize)
xp, sp, bp, terminos_p = agregar_balance(prueba)
prueba.setObjective(terminos_p["inventario"] + terminos_p["atraso"])     # min Σ h·s + π·b
for i in I:
    # Solo en la prueba: sin atraso al final, para que la demanda no "desaparezca"
    prueba += (bp[i][TT[-1]] == 0), f"prueba_sin_atraso_final_{i}"
prueba.solve(SOLVER)
assert pulp.LpStatus[prueba.status] == "Optimal", pulp.LpStatus[prueba.status]

# Solución en tablas (filas = SKU, columnas = semanas)
X = pd.DataFrame({t: {i: xp[i][t].value() for i in I} for t in TT})
S = pd.DataFrame({t: {i: sp[i][t].value() for i in I} for t in TT})
B = pd.DataFrame({t: {i: bp[i][t].value() for i in I} for t in TT})

# Verificación 1: el balance cierra en las 117 celdas (lado izquierdo − lado derecho ≈ 0)
cierre = (pd.concat([s0, S.iloc[:, :-1]], axis=1).values + X.mul(rend, axis=0).values + B.values
          - d.values - S.values - pd.concat([b0, B.iloc[:, :-1]], axis=1).values)
TOL_SOLVER = 1e-3                                           # cajas: tolerancia numérica del solver
assert np.abs(cierre).max() < TOL_SOLVER, "El balance no cierra"

# Verificación 2: lo vendible que se envasa = demanda total − inventario inicial
vendible_neto = (d.sum(axis=1) - s0).clip(lower=0)
assert np.allclose(X.mul(rend, axis=0).sum(axis=1), vendible_neto + S[TT[-1]], atol=TOL_SOLVER), "Producción vendible ≠ demanda neta"

# Verificación 3: el costo es el del óptimo conocido (mantener el inventario inicial mientras dura)
costo_esperado = sum(float(h[i]) * max(0.0, float(s0[i]) - float(d.loc[i, :t].sum())) for i in I for t in TT)
assert abs(pulp.value(prueba.objective) - costo_esperado) < 1.0, "El costo no coincide con el óptimo conocido"

# Verificación 4: sin capacidad que lo obligue, no debería haber atraso
assert B.values.max() < TOL_SOLVER, "La prueba no debería atrasar"

print(f"Prueba del balance: {pulp.LpStatus[prueba.status]} | costo del solver = {pulp.value(prueba.objective):,.0f} CLP "
      f"| costo esperado = {costo_esperado:,.0f} CLP | máx. error de cierre del balance = {np.abs(cierre).max():.1e} cajas")
display(pd.DataFrame({"demanda 13 sem.": d.sum(axis=1), "inventario inicial": s0,
                      "vendible envasado": X.mul(rend, axis=0).sum(axis=1), "cajas envasadas": X.sum(axis=1),
                      "cajas mermadas": X.sum(axis=1) * (1 - rend), "inventario final": S[TT[-1]],
                      "atraso total": B.sum(axis=1)}).round(0))